# Olist Dataset — Schema Understanding

## Objective

The purpose of this notebook is to understand the structural organization of the Olist dataset before designing transformation pipelines and dimensional models.

This analysis focuses on:
- Dataset inventory
- Schema inspection
  * Candidate primary key
  * Candidate foreign keys
- Data granularity
- Table relationships

#### *Import Libraries*

The following libraries are used for:
- loading tabular datasets
- handling file paths
- performing schema inspection and profiling

In [2]:
import pandas as pd
from pathlib import Path

#### *Define Data Source Path*

A centralized base path is defined to avoid hardcoded file locations and improve maintainability across the project structure.

In [3]:
DATA_PATH = Path('../data/raw')

#### *Load Raw Datasets*

Datasets are loaded into a dictionary structure to centralize access and simplify future validation, iteration, and automation tasks.

In [4]:
datasets = {
    "customers": pd.read_csv(DATA_PATH / "olist_customers_dataset.csv"),
    "geolocation": pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv"),
    "order_items": pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv"),
    "order_payments": pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv"),
    "order_reviews": pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv"),
    "orders": pd.read_csv(DATA_PATH / "olist_orders_dataset.csv"),
    "products": pd.read_csv(DATA_PATH / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv"),
    "product_category_translation": pd.read_csv(DATA_PATH / "product_category_name_translation.csv")
}

#### *Validate Dataset Loading*

Initial validation is performed to confirm that all datasets were loaded successfully and to inspect their dimensions.

In [5]:
datasets.keys()

dict_keys(['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_translation'])

---

### Dataset Inventory

First view about:
- Tables
- Number of rows
- Number of columns
- Sizes

In [6]:
def show_dataset_info(raw_datasets):
    sumarized_list  = []
    total_size_mb = 0

    for name, df in raw_datasets.items():
        # calculate memory used to tables in MB
        size_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

        total_size_mb += size_mb

        sumarized_list.append({
            'Dataset': name,
            'Rows': df.shape[0],
            'Columns': df.shape[1],
            'Size (MB)': size_mb
        })

    df_sumarized = pd.DataFrame(sumarized_list)
    # display DataFrame without index
    display(df_sumarized.style.hide(axis='index'))

    # calculate memory used to all dataset (GB)
    total_size_gb = total_size_mb / 1024
    print('-' * 50)
    print(f'Total Size: {total_size_gb:.2f} GB')

In [7]:
show_dataset_info(datasets)

Dataset,Rows,Columns,Size (MB)
customers,99441,5,26.586405
geolocation,1000163,5,129.380595
order_items,112650,7,35.989649
order_payments,103886,5,16.229413
order_reviews,99224,7,39.124777
orders,99441,8,52.937277
products,32951,9,6.296564
sellers,3095,4,0.588048
product_category_translation,71,2,0.008999


--------------------------------------------------
Total Size: 0.30 GB


---

### Initial Schema Inspection

Checks tables to see estructure and what represents of the marketplace

This inspection aims to:
- understand tables structure
- identify important columns
- inspect data types
- detect potential PK and FK

#### Data Catalog:

Understanding the estructure of the all tables.

In [8]:
def sumarize_tables(datasets):
    data_catalog = []

    for df_name, df in datasets.items():
        for col in df.columns:
            # We calculate metrics for the current column
            null_pct = df[col].isna().mean() * 100
            null_count = df[col].isna().sum()
            total_rows = df[col].shape[0]
            unique_values = df[col].nunique()
            duplicated = df[col].duplicated().sum()
            dtype = str(df[col].dtype)

            data_catalog.append({
                'table_name': df_name,
                'column_name': col,
                'dtype': dtype,
                'total_rows': total_rows,
                'unique_values': unique_values,
                'duplicated': duplicated,
                'null_pct': f"{null_pct:.2f}%",
                'null_count': null_count
            })
    datasets_catalog = pd.DataFrame(data_catalog)
    return datasets_catalog

In [9]:
data_tables_catalog = sumarize_tables(datasets)
display(data_tables_catalog.style.hide(axis='index'))

table_name,column_name,dtype,total_rows,unique_values,duplicated,null_pct,null_count
customers,customer_id,object,99441,99441,0,0.00%,0
customers,customer_unique_id,object,99441,96096,3345,0.00%,0
customers,customer_zip_code_prefix,int64,99441,14994,84447,0.00%,0
customers,customer_city,object,99441,4119,95322,0.00%,0
customers,customer_state,object,99441,27,99414,0.00%,0
geolocation,geolocation_zip_code_prefix,int64,1000163,19015,981148,0.00%,0
geolocation,geolocation_lat,float64,1000163,717360,282803,0.00%,0
geolocation,geolocation_lng,float64,1000163,717613,282550,0.00%,0
geolocation,geolocation_city,object,1000163,8011,992152,0.00%,0
geolocation,geolocation_state,object,1000163,27,1000136,0.00%,0


___

### Identifing Potential Primary Keys

In [10]:
def detect_PK(data_catalog_df):
    # Boolean mask of possible PK columns
    p_keys = (data_catalog_df['null_count'] == 0) & (data_catalog_df['unique_values'] == data_catalog_df['total_rows'])

    is_primary_key = data_catalog_df.loc[p_keys, ['table_name', 'column_name']]

    return is_primary_key

In [11]:
is_pk = detect_PK(data_tables_catalog)
display(is_pk.style.hide(axis='index'))

table_name,column_name
customers,customer_id
orders,order_id
orders,customer_id
products,product_id
sellers,seller_id
product_category_translation,product_category_name
product_category_translation,product_category_name_english


### Double-check product_category_translation

- product_category_traslation
  - product_category_name:
    - 71 records
    - 2 orphaned_records

  - product_category_name_english:
    - 71 records
    - 66 orphaned_records

In [12]:
products_values = set(datasets['products']['product_category_name'].dropna().unique())

valores_trans_pdt = set(datasets['product_category_translation']['product_category_name'].unique())
valores_trans_pdt_en = set(datasets['product_category_translation']['product_category_name_english'].unique())

# Dectect the missing values between column PK of 'products' and the candidates of 'product_category_translation'
orphaned_pt = products_values - valores_trans_pdt
orphaned_in = products_values - valores_trans_pdt_en

print(f"{len(products_values)} categories in 'products'")
print(f"{len(valores_trans_pdt)} categories in 'products_translation': \n")

print(f"{len(orphaned_pt)} orphaned if we use Portuguese column")
print(sorted(orphaned_pt))
print(f"{len(orphaned_in)} orphaned if we use English column")

73 categories in 'products'
71 categories in 'products_translation': 

2 orphaned if we use Portuguese column
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
66 orphaned if we use English column


### Double-check geolocation

In [13]:
geolocation_code = (data_tables_catalog['table_name'] == 'geolocation') & (data_tables_catalog['column_name'] == 'geolocation_zip_code_prefix')
geolocation_code_rows = data_tables_catalog.loc[geolocation_code, ['total_rows', 'unique_values']]

display(geolocation_code_rows.style.hide(axis='index'))

total_rows,unique_values
1000163,19015


In [14]:
geo_zip_codes = set(datasets['geolocation']['geolocation_zip_code_prefix'].unique())
customer_zip_codes = set(datasets['customers']['customer_zip_code_prefix'].unique())
seller_zip_codes = set(datasets['sellers']['seller_zip_code_prefix'].unique())

business_zip_codes = customer_zip_codes.union(seller_zip_codes)

matches = business_zip_codes.intersection(geo_zip_codes)
missing_in_geo = business_zip_codes - geo_zip_codes

coverage_percentage = (len(matches) / len(business_zip_codes)) * 100

print(f"{len(business_zip_codes)} Zip codes in (Customers + Sellers)")
print(f"{len(geo_zip_codes)} Zip codes in geolocation")
print(f"{len(matches)} Matches between (customers + sellers) and zip codes in geolocation")
print(f"{len(missing_in_geo)} orphaned zip codes")
print(f"{coverage_percentage:.2f}% of matches")

15078 Zip codes in (Customers + Sellers)
19015 Zip codes in geolocation
14916 Matches between (customers + sellers) and zip codes in geolocation
162 orphaned zip codes
98.93% of matches


---

### Composite PK

In [ ]:
composites_columns = {
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'order_reviews': ['order_id', 'review_id']
}

for tabla, columnas in composites_columns.items():
    duplicados = datasets[tabla].duplicated(subset=columnas).sum()
    print(f"Duplicados en la combinación de {tabla}: {duplicados}")

Duplicados en la combinación de order_items: 0
Duplicados en la combinación de order_payments: 0
Duplicados en la combinación de order_reviews: 0


___

### Identifing Foreing Keys:

In [19]:

def validate_foreign_key(df_child, col_child, df_parent, col_parent):

    child_set = set(df_child[col_child].dropna().unique())
    parent_set = set(df_parent[col_parent].dropna().unique())

    orphans = child_set - parent_set
    is_fk = len(orphans) == 0

    total_child_keys = len(child_set)
    matches = len(child_set.intersection(parent_set))
    coverage_pct = (matches / total_child_keys * 100) if total_child_keys > 0 else 0

    return {
        'is_fk_perfect': is_fk,
        'child_unique_keys': total_child_keys,
        'matches': matches,
        'orphans_count': len(orphans),
        'coverage_percentage': round(coverage_pct, 2),
        'sample_orphans': list(orphans)[:5] # Te muestra un botón de muestra si falla
    }


In [ ]:
relationships_to_test = [
    ('orders', 'customer_id', 'customers', 'customer_id'),
    ('order_items', 'order_id', 'orders', 'order_id'),
    ('order_items', 'product_id', 'products', 'product_id'),
    ('order_items', 'seller_id', 'sellers', 'seller_id'),
    ('products', 'product_category_name', 'product_category_translation', 'product_category_name'),
    ('customers', 'customer_zip_code_prefix', 'geolocation', 'geolocation_zip_code_prefix'),
    ('sellers', 'seller_zip_code_prefix', 'geolocation', 'geolocation_zip_code_prefix'),
]

fk_reports = []

for child_t, child_c, parent_t, parent_c in relationships_to_test:
    result = validate_foreign_key(
        datasets[child_t], child_c,
        datasets[parent_t], parent_c
    )

    # Añadimos los nombres de las tablas para el reporte final
    result['relationship'] = f"{child_t}.{child_c} -> {parent_t}.{parent_c}"
    fk_reports.append(result)

df_fk_summary = pd.DataFrame(fk_reports)
display(df_fk_summary[['relationship', 'is_fk_perfect', 'coverage_percentage', 'orphans_count', 'sample_orphans']])

,relationship,is_fk_perfect,coverage_percentage,orphans_count,sample_orphans
0,orders.customer_id -> customers.customer_id,True,100.00,0,[]
1,order_items.order_id -> orders.order_id,True,100.00,0,[]
2,order_items.product_id -> products.product_id,True,100.00,0,[]
3,order_items.seller_id -> sellers.seller_id,True,100.00,0,[]
4,products.product_category_name -> product_cate...,False,97.26,2,[portateis_cozinha_e_preparadores_de_alimentos...
5,customers.customer_zip_code_prefix -> geolocat...,False,98.95,157,"[28160, 56327, 75784, 29196, 71698]"
6,sellers.seller_zip_code_prefix -> geolocation....,False,99.69,7,"[72580, 37708, 2285, 7412, 82040]"


---

### Validate Table Granularity

The granularity is inspected to determine business event representation.

- table relationships
- dimensional modeling

---

### Design Decisions & Architectural Schema Summary

* **Candidate Primary Keys:**
    * Table `customers`: Column `customer_id`
    * Table `orders`: Column `order_id`
    * Table `products`: Column `product_id`
    * Table `sellers`: Column `seller_id`
    * Table `product_category_translation`: Column `product_category_name`

* **Candidate Foreign Key:**
    * Table `orders`: Column `customer_id` (References `customers.customer_id`)
    * Table `order_items`: Column `order_id` (References `orders.order_id`)
    * Table `order_items`: Column `product_id` (References `products.product_id`)
    * Table `order_items`: Column `seller_id` (References `sellers.seller_id`)

* **Composite Keys Required:**
    * Table `order_items`: Columns `order_id + order_item_id`
    * Table  `order_payments`: Columns `order_id + payment_sequential`

- orders:
  - `order_id`: candidate primary key (100% unique, no null values).
  - `customer_id`: candidate foreing key, as it connects to the customers table

- product_category_translation:
  - `product_category_name`: candidate primary key (100% unique, no null values). 2 orphaned records detected, this represents a minor data integrity issue where some products lack an English translation
   - `product_category_name_english`: candidate key (100% unique, no null values). It will be taken as False PK.
  - Action Plan: To avoid unexpected NaN values during data merging, a defensive transformation (e.g., .fillna() or record injection) will be implemented during the Transformation/ETL phase of the pipeline

- geolocation:
  - Any column were identified in geolocation table as primary key candidate (they contains duplicated values, which it is expected)
  - Apply mean grouping on geolocation_zip_code_prefix to resolve duplicates and compute regional centroids.
  - Append 162 placeholder records with NaN coordinates for missing business zip codes.